# 🌌 Sovereign Audio Intelligence: Live Mastering & Audit Notebook

This notebook provides a complete interface to run the **Sovereign V2 Neural Mastering Engine** with Gemma-recommended mathematical improvements and execute the **Gemma 4 Large Audit** loop.

## 🛠️ Startup Guide & System Architecture

### 📡 Active Ray Swarm Deployments (Port `8000`)
- **`SovereignNeuralRelay`** (`/neural_relay`): Orchestrates DNA $\rightarrow$ Latent $\rightarrow$ Param $\rightarrow$ Render.
- **`SovereignDNAEngine`** (`/sovereign-brain`): Standard DNA projection service.
- **`SovereignSieve`** (`/`): The 24-replica style classifier.
- **`AudioLLM`** (`/audio-llm`): The Audio LLM inference node.

### 📂 System Hard Paths
- **Production Model**: `C:\WEB CASE STUDY\sovereign_big_brain_exhaustive.onnx`
- **External Weights**: `C:\WEB CASE STUDY\sovereign_big_brain_exhaustive.onnx.data`
- **Gold Baseline (LanceDB)**: `C:\STUDIES_BACKUP\vectors\lancedb_omni_snowflake_rag`
- **Output Masters (V2)**: `C:\WEB CASE STUDY\sovereign_onnx_masters_v2`
- **Audit Reports**: `C:\WEB CASE STUDY\logs`

## 🧬 Cell 1: Enhanced V2 Mastering Engine (With Gemma Optimizations)

This cell runs the V2 Mastering process on `SCAR-red strobe.mp3` with two critical mathematical optimizations:
1. **`PchipInterpolator`** replaces `interp1d(kind='cubic')` to guarantee monotonicity and prevent parameter overshoot.
2. **Micro-Dynamics Resolution**: Window size is reduced to **0.5 seconds** and hop size to **0.1 seconds** (originally 2s and 0.5s) to capture rapid transients.
3. **WAV Output**: Automatically converts output format to `.wav` (since `soundfile` cannot write `.mp3`), stamping the file with a unique timestamp.

In [1]:
import os
import sys
import time
import datetime
import numpy as np
import librosa
import soundfile as sf
import onnxruntime as ort
import lancedb
from sklearn.preprocessing import StandardScaler
from scipy.signal import butter, sosfilt
from scipy.interpolate import PchipInterpolator  # Optimized monotonicity
from pedalboard import Pedalboard, Compressor, Gain, Limiter, HighShelfFilter, PeakFilter
from pathlib import Path
import json

# --- Paths ---
INPUT_TRACK = r"C:\Users\adams\Downloads\SCAR-red strobe.mp3"
MODEL_PATH = r"C:\WEB CASE STUDY\sovereign_big_brain_exhaustive.onnx"
BASELINE_DB = r"C:\STUDIES_BACKUP\vectors\lancedb_omni_snowflake_rag"
OUTPUT_DIR = r"C:\WEB CASE STUDY\sovereign_onnx_masters_v2"
CURVES_DIR = r"C:\WEB CASE STUDY\curves_onnx_v2"
MONO_CROSSOVER_HZ = 150.0

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CURVES_DIR, exist_ok=True)

TARGET_COLS = [
    "rms", "crest_factor", "sub_bass_energy", "bass_energy", "mid_energy",
    "high_energy", "spectral_centroid", "spectral_bandwidth",
    "spectral_rolloff", "spectral_flatness", "spectral_contrast", "zero_crossing_rate"
]

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
fname = os.path.basename(INPUT_TRACK)
base_name, _ = os.path.splitext(fname)
# Force output extension to .wav to prevent sf.write failure
out_audio_path = os.path.join(OUTPUT_DIR, f"ONNX2_{timestamp}_{base_name}.wav")
out_json_path = out_audio_path + ".dna.json"

print(f"[1/5] Loading Gold Baseline from LanceDB...")
db = lancedb.connect(BASELINE_DB)
df_gold = db.open_table("omni_semantic_baselines").to_pandas()
mask = df_gold["track_name"].str.contains("chris lake|somebody", case=False, na=False)
df_gold = df_gold[mask].reset_index(drop=True)

scaler = StandardScaler()
scaler.fit(df_gold[TARGET_COLS].fillna(0.0).values.astype(np.float32))
gold_mean_dict = {col: float(df_gold[col].mean()) for col in TARGET_COLS}

print(f"[2/5] Loading ONNX Session...")
session = ort.InferenceSession(MODEL_PATH)
input_name = session.get_inputs()[0].name

def extract_dna(y_mono, sr, n_fft=2048, hop=512):
    rms_val = float(np.sqrt(np.mean(y_mono ** 2)))
    peak = float(np.max(np.abs(y_mono))) + 1e-9
    crest = peak / (rms_val + 1e-9)
    S = np.abs(librosa.stft(y_mono, n_fft=n_fft, hop_length=hop))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    centroid = float(np.mean(librosa.feature.spectral_centroid(S=S, freq=freqs)))
    bandwidth = float(np.mean(librosa.feature.spectral_bandwidth(S=S, freq=freqs)))
    rolloff = float(np.mean(librosa.feature.spectral_rolloff(S=S, freq=freqs)))
    flatness = float(np.mean(librosa.feature.spectral_flatness(S=S)))
    contrast = float(np.mean(librosa.feature.spectral_contrast(S=S, sr=sr)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y_mono)))
    
    def band_e(S, f, lo, hi):
        m = (f >= lo) & (f < hi)
        return float(np.mean(S[m, :] ** 2)) if m.any() else 0.0
        
    sub_bass = band_e(S, freqs, 20, 60)
    bass = band_e(S, freqs, 60, 250)
    mid = band_e(S, freqs, 250, 4000)
    high = band_e(S, freqs, 4000, 20000)
    
    f12 = np.array([rms_val, crest, sub_bass, bass, mid, high,
                    centroid, bandwidth, rolloff, flatness, contrast, zcr], dtype=np.float32)
    d64 = np.zeros(64, dtype=np.float32)
    d64[:12] = f12
    return d64, f12

def predict(dna_64):
    norm = session.run(None, {input_name: dna_64.reshape(1, 64).astype(np.float32)})[0][0]
    real = scaler.inverse_transform(norm.reshape(1, -1))[0]
    return {col: float(real[i]) for i, col in enumerate(TARGET_COLS)}

print(f"[3/5] Loading track: {fname}...")
y, sr = librosa.load(INPUT_TRACK, sr=None, mono=False)
if y.ndim == 1:
    y = y[np.newaxis, :]
is_stereo = y.shape[0] >= 2
y_mono = librosa.to_mono(y) if is_stereo else y[0]

print(f"[4/5] Extracting dynamic trajectories (Window=0.5s, Hop=0.1s)...")
duration = len(y_mono) / sr
points = []
for start in np.arange(0, max(duration - 0.5, 0.05), 0.1):
    s, e = int(start * sr), int(min((start + 0.5) * sr, len(y_mono)))
    chunk = y_mono[s:e]
    if len(chunk) < 512:
        continue
    d64, f12 = extract_dna(chunk, sr)
    points.append((start, d64, f12))

times_arr = np.array([p[0] for p in points])
all_params = [predict(p[1]) for p in points]

curves = {}
for col in TARGET_COLS:
    vals = np.array([p[col] for p in all_params])
    # PchipInterpolator guarantees shape preservation and prevents overshooting
    curves[col] = PchipInterpolator(times_arr, vals)

print(f"[5/5] Processing mastering effects stream...")
block_samples = int(0.1 * sr)  # Smaller block processing
mastered = np.copy(y)

for start_samp in range(0, y.shape[1] - block_samples, block_samples):
    end_samp = start_samp + block_samples
    t = np.clip(start_samp / sr, times_arr[0], times_arr[-1])
    p = {col: float(curves[col](t)) for col in TARGET_COLS}
    chunk = y[:, start_samp:end_samp].copy()

    chunk_rms = float(np.sqrt(np.mean(chunk ** 2))) + 1e-9
    target_rms = max(p["rms"], 1e-6)
    blended_rms = target_rms * 0.7 + gold_mean_dict["rms"] * 0.3
    gain_db = np.clip(20 * np.log10(blended_rms + 1e-9) - 20 * np.log10(chunk_rms), -6.0, 6.0)

    ratio = np.clip(p["crest_factor"] * 0.6, 1.2, 4.0)
    threshold = np.clip(-10.0 - abs(gain_db) * 0.5, -24.0, -6.0)

    mid_delta = 0.0
    if gold_mean_dict["mid_energy"] > 0 and p["mid_energy"] > 0:
        mid_delta = np.clip(-3.0 * np.log2(p["mid_energy"] / gold_mean_dict["mid_energy"] + 1e-9), -3.0, 3.0)
    high_delta = 0.0
    if gold_mean_dict["high_energy"] > 0 and p["high_energy"] > 0:
        high_delta = np.clip(-2.0 * np.log2(p["high_energy"] / gold_mean_dict["high_energy"] + 1e-9), -2.0, 3.0)

    board = Pedalboard([
        Gain(gain_db=float(gain_db)),
        PeakFilter(cutoff_frequency_hz=1000.0, gain_db=float(mid_delta), q=0.7),
        HighShelfFilter(cutoff_frequency_hz=8000.0, gain_db=float(high_delta), q=0.7),
        Compressor(threshold_db=float(threshold), ratio=float(ratio), attack_ms=15.0, release_ms=120.0),
        Limiter(threshold_db=-1.0, release_ms=150.0),
    ])
    mastered[:, start_samp:end_samp] = board(chunk, sample_rate=sr)

if is_stereo and mastered.shape[0] >= 2:
    sos_low = butter(4, MONO_CROSSOVER_HZ, btype='low', fs=sr, output='sos')
    sos_high = butter(4, MONO_CROSSOVER_HZ, btype='high', fs=sr, output='sos')
    low_mono = (sosfilt(sos_low, mastered[0]) + sosfilt(sos_low, mastered[1])) * 0.5
    mastered[0] = low_mono + sosfilt(sos_high, mastered[0])
    mastered[1] = low_mono + sosfilt(sos_high, mastered[1])

mastered = np.clip(mastered, -1.0, 1.0)
sf.write(out_audio_path, mastered.T, sr)

sidecar = {
    "source": fname,
    "model": "sovereign_big_brain_exhaustive.onnx",
    "version": "v2_pchip_highres",
    "stereo": is_stereo,
    "duration_sec": round(y.shape[1] / sr, 2),
    "inference_count": len(points),
    "audio_output_path": out_audio_path,
    "timestamp": timestamp
}

with open(out_json_path, "w") as f:
    json.dump(sidecar, f, indent=2)

print(f"\n✅ MASTERING COMPLETE!")
print(f"Audio Mastered File: {out_audio_path}")
print(f"DNA Sidecar File: {out_json_path}")

[1/5] Loading Gold Baseline from LanceDB...
[2/5] Loading ONNX Session...
[3/5] Loading track: SCAR-red strobe.mp3...
[4/5] Extracting dynamic trajectories (Window=0.5s, Hop=0.1s)...
[5/5] Processing mastering effects stream...

✅ MASTERING COMPLETE!
Audio Mastered File: C:\WEB CASE STUDY\sovereign_onnx_masters_v2\ONNX2_20260716_220127_SCAR-red strobe.wav
DNA Sidecar File: C:\WEB CASE STUDY\sovereign_onnx_masters_v2\ONNX2_20260716_220127_SCAR-red strobe.wav.dna.json


## 🔬 Cell 2: Gemma 4 Large System Audit (Grounded Reasoning)

This cell packages the entire context (original unmastered mixdown details, targets, C++ DSP code structures, and the **newly mastered track JSON** generated by Cell 1) and queries **Gemma 4 Large** (`gemma-4-31b-it`) for an audit analysis of the changes.

In [ ]:
import os
import sys
from google import genai

timestamp = globals().get('timestamp', 'current')
json_path = globals().get('out_json_path', r"C:\WEB CASE STUDY\sovereign_onnx_masters_v2\ONNX2_SCAR-red strobe.mp3.dna.json")
gemma_out_file = os.path.join(r"C:\WEB CASE STUDY\logs", f"gemma_system_audit_{timestamp}.md")

def safe_read_file(path, max_chars=30000):
    if not os.path.exists(path):
        return f"[File not found: {path}]"
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
            if len(content) > max_chars:
                return content[-max_chars:] + "\n... [TRUNCATED] ..."
            return content
    except Exception as e:
        return f"[Error reading {path}: {e}]"

print(f"Loading dynamic JSON telemetry from: {json_path}")
raw_mixdown_metrics = """
File: C:\Users\adams\Downloads\SCAR-red strobe.mp3
Starting RMS: 0.0740 (-22.61 dB)
Starting Crest Factor: 7.327
"""

target_baseline_genre = """
Tech House standard (Chris Lake reference):
  - Sub + Bass dominate ~88.9% of total spectrum
  - Mids/Highs kept clean and sparse
"""

mastering_json_data = safe_read_file(json_path)
dsp_code = safe_read_file(r"C:\WEB CASE STUDY\sovereign_onnx_remaster_v2.py")
swarm_log = safe_read_file(r"C:\WEB CASE STUDY\logs\ray_code_swarm_boot.log", max_chars=10000)

prompt = f"""
=== SOVEREIGN SYSTEM AUDIT AND AUDIT REPORT ===

You are the Sovereign Council Reasoner (Model: gemma-4-31b-it).
Below is the complete context of the mastering run, baseline targets, system logs, and C++ Python code variables.

--- 1. INPUT MIXDOWN FEATURES (BEFORE RAY) ---
{raw_mixdown_metrics}

--- 2. GENRE REF BASELINES (BEFORE RAY) ---
{target_baseline_genre}

--- 3. COMPLETE MASTERING DATA JSON ---
{mastering_json_data}

--- 4. DSP ENGINE CODE ---
{dsp_code}

--- 5. RAY SWARM RUN LOGS ---
{swarm_log}

==================================================
MISSION AND QUESTIONS FOR THE COUNCIL:
1. Provide a comprehensive summary of how to improve the system code and logs.
2. Explain how you think the mastering process works from start to finish based on the actual code, features, and database baselines.
3. Decipher the exact data path from unmastered input to final v2 output.
==================================================
"""

print("Connecting to GenAI Client...")
client = genai.Client()

print(f"Querying gemma-4-31b-it (Streaming response to {gemma_out_file})...")
try:
    response = client.models.generate_content(
        model="gemma-4-31b-it",
        contents=prompt
    )
    print("\n=== RESPONSE ===")
    print(response.text)
    
    with open(gemma_out_file, "w", encoding="utf-8") as f:
        f.write(response.text)
    print(f"\nSaved gemma-4 audit report successfully.")
except Exception as e:
    print(f"Error: {e}")

## 💻 Cell 3: Code Execution Sandbox (Gemini 2.5 Flash Interpreter)

This cell connects to the **Gemini 2.5 Flash Python Interpreter Sandbox** to run numeric range and scaling checks on the **newly remastered track's DNA JSON** generated in Cell 1.

In [ ]:
import os
import sys
import json
import onnxruntime as ort
from google import genai
from google.genai import types

onnx_path = r"C:\WEB CASE STUDY\sovereign_big_brain_exhaustive.onnx"
timestamp = globals().get('timestamp', 'current')
json_path = globals().get('out_json_path', r"C:\WEB CASE STUDY\sovereign_onnx_masters_v2\ONNX2_SCAR-red strobe.mp3.dna.json")
gemma_sandbox_out = os.path.join(r"C:\WEB CASE STUDY\logs", f"gemma_sandbox_audit_report_{timestamp}.md")

print(f"Loading metadata for model and dynamic master JSON from: {json_path}")
session = ort.InferenceSession(onnx_path)
onnx_info = {
    "inputs": [{"name": i.name, "shape": i.shape, "type": i.type} for i in session.get_inputs()],
    "outputs": [{"name": o.name, "shape": o.shape, "type": o.type} for o in session.get_outputs()]
}

raw_json_data = "{}"
if os.path.exists(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        raw_json_data = f.read()
else:
    # Fallback to general master if cell 1 was not run in this session
    fallback = r"C:\WEB CASE STUDY\sovereign_onnx_masters_v2\ONNX2_SCAR-red strobe.mp3.dna.json"
    if os.path.exists(fallback):
        with open(fallback, "r", encoding="utf-8") as f:
            raw_json_data = f.read()

prompt = f"""
You are the Sovereign Council Reasoner. You have access to a Python Code Execution Sandbox.
We want to verify the dynamic range scaling equations and check shape compatibility for our ONNX model.

Here is the local metadata from our ONNX model:
{json.dumps(onnx_info, indent=2)}

Below is the complete raw telemetry JSON file containing the actual array of target values predicted by the model:
{raw_json_data}

Unmastered Reference:
- Unmastered RMS for SCAR-red strobe: 0.0740 (-22.61 dB)
- Gold Chris Lake Baseline Target Mean RMS: 0.3160 (-10.01 dB)

TASK:
1. Write a Python script to parse the provided raw JSON data in your sandbox. Extract the list of model-predicted RMS targets.
2. In the sandbox, use NumPy to calculate:
   - The mean, variance, and standard deviation of the predicted targets.
   - The makeup gain curve using the formula: gain_db = 20 * log10(target_rms) - 20 * log10(0.0740)
3. Check the calculated gain values:
   - Identify if any values exceed safe boundaries (e.g., gain > 18dB or gain < -12dB).
   - Propose an optimized clipping or blending formula (such as blending 70% target with 30% baseline) to prevent transient distortion.
4. Output the sandbox execution print logs and your final analysis.
"""

client = genai.Client()
print("Sending query to Gemini 2.5 Flash with Code Execution sandbox tool...")
try:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(code_execution=types.ToolCodeExecution())]
        )
    )
    print("\n=== SANDBOX REPORT ===")
    print(response.text)
    
    with open(gemma_sandbox_out, "w", encoding="utf-8") as f:
        f.write(response.text)
    print(f"\nSaved sandbox report successfully.")
except Exception as e:
    print(f"Error: {e}")

## ⚡ Cell 4: Native C++ High-Performance Rendering Engine (LLVM Clang / CMake)

This cell compiles and executes the **Sovereign Native C++ DSP Rendering Engine** (`sovereign_render.cpp`) built with **LLVM Clang** and **Ninja**. It renders audio trajectory curves at native C++ memory speed with sample-accurate slew-rate smoothing.

In [ ]:
import os
import subprocess

cpp_source = r"C:\WEB CASE STUDY\sovereign_render.cpp"
cmake_lists = r"C:\WEB CASE STUDY\CMakeLists.txt"
curves_file = r"C:\WEB CASE STUDY\test_curves.csv"
input_audio = r"C:\WEB CASE STUDY\sovereign_30s_capture.wav"

print("[1/3] Verifying C++ Source and CMake Toolchain...")
assert os.path.exists(cpp_source), f"Missing C++ source: {cpp_source}"
assert os.path.exists(cmake_lists), f"Missing CMakeLists.txt: {cmake_lists}"

print("[2/3] Building C++ Binary with CMake + Ninja (Clang)... ")
build_cmd = ["wsl", "bash", "-c", "cd '/mnt/c/WEB CASE STUDY' && CC=clang CXX=clang++ cmake -B build_wsl -G Ninja && cmake --build build_wsl"]
res = subprocess.run(build_cmd, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("Build Error:", res.stderr)

print("[3/3] Executing Native C++ Audio Rendering Engine...")
run_cmd = ["wsl", "bash", "-c", f"cd '/mnt/c/WEB CASE STUDY' && ./build_wsl/SovereignAudioIngest input.wav {os.path.basename(curves_file)} output.wav"]
run_res = subprocess.run(run_cmd, capture_output=True, text=True)
print(run_res.stdout)


## ⚡ Cell 4: Native C++ High-Performance Rendering Engine (LLVM Clang / CMake)

This cell compiles and executes the **Sovereign Native C++ DSP Rendering Engine** (`sovereign_render.cpp`) built with **LLVM Clang** and **Ninja**. It renders audio trajectory curves at native C++ memory speed with sample-accurate slew-rate smoothing.

In [ ]:
import os
import subprocess

cpp_source = r"C:\WEB CASE STUDY\sovereign_render.cpp"
cmake_lists = r"C:\WEB CASE STUDY\CMakeLists.txt"
curves_file = r"C:\WEB CASE STUDY\test_curves.csv"
input_audio = r"C:\WEB CASE STUDY\sovereign_30s_capture.wav"

print("[1/3] Verifying C++ Source and CMake Toolchain...")
assert os.path.exists(cpp_source), f"Missing C++ source: {cpp_source}"
assert os.path.exists(cmake_lists), f"Missing CMakeLists.txt: {cmake_lists}"

print("[2/3] Building C++ Binary with CMake + Ninja (Clang)... ")
build_cmd = ["wsl", "bash", "-c", "cd '/mnt/c/WEB CASE STUDY' && CC=clang CXX=clang++ cmake -B build_wsl -G Ninja && cmake --build build_wsl"]
res = subprocess.run(build_cmd, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("Build Error:", res.stderr)

print("[3/3] Executing Native C++ Audio Rendering Engine...")
run_cmd = ["wsl", "bash", "-c", f"cd '/mnt/c/WEB CASE STUDY' && ./build_wsl/SovereignAudioIngest input.wav {os.path.basename(curves_file)} output.wav"]
run_res = subprocess.run(run_cmd, capture_output=True, text=True)
print(run_res.stdout)
